In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_selection import SelectFromModel, RFE
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('data/modified/dirty_model.csv')
df.head()   

,First_pokemon,Second_pokemon,Winner,Name_P1,Type 1_P1,Type 2_P1,Type_P1,Abilities_P1,HiddenAbility_P1,Generation_P1,...,DamageFromSteel_P2,DamageFromFire_P2,DamageFromWater_P2,DamageFromGrass_P2,DamageFromElectric_P2,DamageFromPsychic_P2,DamageFromIce_P2,DamageFromDragon_P2,DamageFromDark_P2,DamageFromFairy_P2
0,266,298,298,Larvitar,Rock,Ground,"['Rock', 'Ground']",['Guts'],['Sand Veil'],II,...,1.0,2.0,0.5,0.5,0.5,0.0,2.0,1.0,0.5,2.0
1,702,701,701,Virizion,Grass,Fighting,"['Grass', 'Fighting']",['Justified'],[],V,...,2.0,0.5,2.0,2.0,1.0,2.0,1.0,1.0,0.5,2.0
2,151,231,151,Omastar,Rock,Water,"['Rock', 'Water']","['Swift Swim', 'Shell Armor']",['Weak Armor'],I,...,2.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,657,752,657,Joltik,Bug,Electric,"['Bug', 'Electric']","['Compound Eyes', 'Unnerve']",['Swarm'],V,...,0.5,2.0,1.0,0.5,1.0,0.5,0.5,0.5,2.0,0.5
4,192,134,134,Natu,Psychic,Flying,"['Psychic', 'Flying']","['Synchronize', 'Early Bird']",['Magic Bounce'],II,...,2.0,2.0,1.0,1.0,1.0,0.5,0.5,1.0,2.0,1.0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10070 entries, 0 to 10069
Data columns (total 99 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   First_pokemon          10070 non-null  int64  
 1   Second_pokemon         10070 non-null  int64  
 2   Winner                 10070 non-null  int64  
 3   Name_P1                10070 non-null  object 
 4   Type 1_P1              10070 non-null  object 
 5   Type 2_P1              10070 non-null  object 
 6   Type_P1                10070 non-null  object 
 7   Abilities_P1           10070 non-null  object 
 8   HiddenAbility_P1       10070 non-null  object 
 9   Generation_P1          10070 non-null  object 
 10  Hp_P1                  10070 non-null  float64
 11  Attack_P1              10070 non-null  float64
 12  Defense_P1             10070 non-null  float64
 13  SpecialAttack_P1       10070 non-null  float64
 14  SpecialDefense_P1      10070 non-null  float64
 15  Sp

In [4]:
# type1dum = pd.get_dummies(df['Type_P1'], prefix='Type', dtype=int)
# type2dum = pd.get_dummies(df['Type_P2'], prefix='Type', dtype=int)

# dummies_total = type1dum.add(type2dum, fill_value=0)

# print(dummies_total)

# df = pd.concat([df, dummies_total], axis=1)
# df = df.drop(['Type_P1', 'Type_P2'], axis=1)

# egggroup1dum = pd.get_dummies(df['EggGroup_P1'], prefix='Egg', dtype=int)
# egggroup2dum = pd.get_dummies(df['EggGroup_P2'], prefix='Egg', dtype=int)
# dummies_total = egggroup1dum.add(egggroup2dum, fill_value=0)

# print(dummies_total)

# df = pd.concat([df, dummies_total], axis=1)
# df = df.drop(['EggGroup_P1', 'EggGroup_P2'], axis=1)

In [5]:
# leveling_map = {
#     'Slow': 1250000,
#     'Medium Slow': 1059860,
#     'Medium Fast': 1000000,
#     'Fast': 800000,
#     'Erratic': 600000,
#     'Fluctuating': 1640000
# }

# # Áp dụng replace và sau đó chuyển kiểu rõ ràng sang float
# df['LevelingRate_P1'] = df['LevelingRate_P1'].replace(leveling_map).astype(float)
# df['LevelingRate_P2'] = df['LevelingRate_P2'].replace(leveling_map).astype(float)

In [6]:
df['Winner'] = np.where(df['Winner'] == df['First_pokemon'], 0, 1)

In [7]:
obj_cols = df.select_dtypes(include=['object']).columns
obj_cols

Index(['Name_P1', 'Type 1_P1', 'Type 2_P1', 'Type_P1', 'Abilities_P1',
       'HiddenAbility_P1', 'Generation_P1', 'GenderProbM_P1', 'Category_P1',
       'EggGroup_P1', 'LevelingRate_P1', 'PreevoName_P1', 'Name_P2',
       'Type 1_P2', 'Type 2_P2', 'Type_P2', 'Abilities_P2', 'HiddenAbility_P2',
       'Generation_P2', 'GenderProbM_P2', 'Category_P2', 'EggGroup_P2',
       'LevelingRate_P2', 'PreevoName_P2'],
      dtype='object')

In [8]:
df = df.drop(columns=obj_cols)

In [9]:
X = df.drop(columns=['First_pokemon', 'Second_pokemon', 'Winner'])
y = df['Winner']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=42)

In [ ]:
rf_estimator = RandomForestClassifier(n_estimators=100, random_state=42)
rfe_selector = RFE(estimator=rf_estimator, n_features_to_select=20, step=1)

print("Đang thực hiện Feature Selection...")
rfe_selector.fit(X_train, y_train)

selected_features = X.columns[rfe_selector.support_]
print(f"Các feature được chọn ({len(selected_features)}): {list(selected_features)}")

X_train_selected = rfe_selector.transform(X_train)
X_val_selected = rfe_selector.transform(X_val)
X_test_selected = rfe_selector.transform(X_test)

Đang thực hiện Feature Selection...
Các feature được chọn (20): ['Hp_P1', 'Attack_P1', 'Defense_P1', 'SpecialAttack_P1', 'SpecialDefense_P1', 'Speed_P1', 'TotalStats_P1', 'Weight_P1', 'Height_P1', 'CatchRate_P1', 'Hp_P2', 'Attack_P2', 'Defense_P2', 'SpecialAttack_P2', 'SpecialDefense_P2', 'Speed_P2', 'TotalStats_P2', 'Weight_P2', 'Height_P2', 'CatchRate_P2']


In [ ]:
print("\n--- Đang huấn luyện mô hình tối ưu ---")
rf_final = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

rf_final.fit(X_train_selected, y_train) 

train_acc = accuracy_score(y_train, rf_final.predict(X_train_selected))
val_acc = accuracy_score(y_val, rf_final.predict(X_val_selected))
test_acc = accuracy_score(y_test, rf_final.predict(X_test_selected))

print(
    f"Train: {X_train_selected.shape} "
    f"Val: {X_val_selected.shape} "
    f"Test: {X_test_selected.shape}"
)

print("=== Random Forest accuracy ===") 

print(f"Train: {train_acc:.4f}")
print(f"Val: {val_acc:.4f}")
print(f"Test: {test_acc:.4f}")


--- Đang huấn luyện mô hình tối ưu ---
Train: (6847, 20) Val: (1209, 20) Test: (2014, 20)
=== Random Forest accuracy ===
Train: 1.0000
Val: 0.9256
Test: 0.9166


In [15]:
gb_estimator = GradientBoostingClassifier(n_estimators=100, random_state=42)

rfe_selector = RFE(estimator=gb_estimator, n_features_to_select=20, step=1)

print("Đang thực hiện Feature Selection...")

rfe_selector.fit(X_train, y_train) 

selected_features = X.columns[rfe_selector.support_]
print(f"Các feature được chọn ({len(selected_features)}): {list(selected_features)}")

X_train_selected = rfe_selector.transform(X_train)
X_val_selected = rfe_selector.transform(X_val)
X_test_selected = rfe_selector.transform(X_test)

Đang thực hiện Feature Selection...
Các feature được chọn (20): ['Hp_P1', 'Attack_P1', 'Defense_P1', 'SpecialAttack_P1', 'Speed_P1', 'TotalStats_P1', 'Weight_P1', 'DamageFromGround_P1', 'DamageFromPsychic_P1', 'Hp_P2', 'Attack_P2', 'Defense_P2', 'SpecialDefense_P2', 'Speed_P2', 'TotalStats_P2', 'CatchRate_P2', 'DamageFromPoison_P2', 'DamageFromElectric_P2', 'DamageFromPsychic_P2', 'DamageFromDark_P2']


In [16]:
print("\nĐang train model Gradient Boosting trên các feature đã chọn...")
final_model = GradientBoostingClassifier(
    n_estimators=200, 
    learning_rate=0.1, 
    max_depth=3, 
    random_state=42
)

final_model.fit(X_train_selected, y_train) 

train_acc = accuracy_score(y_train, final_model.predict(X_train_selected))
val_acc = accuracy_score(y_val, final_model.predict(X_val_selected))
test_acc = accuracy_score(y_test, final_model.predict(X_test_selected))

print("\n--- KẾT QUẢ ĐÁNH GIÁ ---")
print(
    f"Train: {X_train_selected.shape} "
    f"Val: {X_val_selected.shape} "
    f"Test: {X_test_selected.shape}"
)

print("=== Gradient Boosting accuracy ===") 

print(f"Train: {train_acc:.4f}")
print(f"Val: {val_acc:.4f}")
print(f"Test: {test_acc:.4f}")


Đang train model Gradient Boosting trên các feature đã chọn...

--- KẾT QUẢ ĐÁNH GIÁ ---
Train: (6847, 20) Val: (1209, 20) Test: (2014, 20)
=== Gradient Boosting accuracy ===
Train: 0.9382
Val: 0.9165
Test: 0.9121
